In [1]:
import numpy as np
import pandas as pd

from PyFairnessAI.model_selection import RandomizedSearchCVFairness
from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

from sklearn.model_selection import train_test_split, StratifiedKFold

from aif360.datasets import GermanDataset

from PyMachineLearning.models import LogisticRegressionThreshold

pd.set_option('display.max_colwidth', None)

pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
pip install 'aif360[FairAdapt]'
pip install 'aif360[LFR]'


In [2]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [3]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
# The sensitive variable/s must index the df in order to avoid problems with some aif360 processors (PostProcessingMetaEstimator)
german_data_pd.index = german_data_pd[sens_variable_name] 
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
age,,,,,,,,,,,,,,,,,,,,,
1.0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
0.0,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1.0,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [4]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=111)
log_reg_estimator = LogisticRegressionThreshold(solver='liblinear', random_state=123, max_iter=200)

In [6]:
param_grid = {
    'penalty': ['l1', 'l2'], 
    'C': [0.01, 0.1, 1, 10, 30, 50, 75, 100],  # Regularization strength
    'class_weight': ['balanced', None],
    'threshold': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

In [7]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='fairness', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=50, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [8]:
fairness_random_search.cv_results_.head()

,params,predictive-score,fairness-score,combined-score
24,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
4,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
17,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
33,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
27,"{'penalty': 'l2', 'C': 0.01, 'class_weight': 'balanced', 'threshold': 0.8}",0.503361,0.005462,0.559204


In [9]:
fairness_random_search.cv_results_.tail()

,params,predictive-score,fairness-score,combined-score
13,"{'penalty': 'l2', 'C': 1, 'class_weight': None, 'threshold': 0.7}",0.717087,0.278647,0.487010
2,"{'penalty': 'l2', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.4}",0.707283,0.283003,0.460931
37,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced', 'threshold': 0.5}",0.713445,0.283352,0.471951
31,"{'penalty': 'l1', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.4}",0.709804,0.287279,0.458245
18,"{'penalty': 'l1', 'C': 100, 'class_weight': 'balanced', 'threshold': 0.4}",0.709804,0.287279,0.458245


In [10]:
fairness_random_search.best_params_

{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}

In [11]:
fairness_random_search.best_score_

0.0

In [12]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='predictive', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=50, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [13]:
fairness_random_search.cv_results_.head()

,params,predictive-score,fairness-score,combined-score
7,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced', 'threshold': 0.6}",0.731933,0.265248,0.538344
42,"{'penalty': 'l1', 'C': 100, 'class_weight': None, 'threshold': 0.8}",0.731373,0.268584,0.531481
46,"{'penalty': 'l2', 'C': 100, 'class_weight': None, 'threshold': 0.8}",0.729412,0.265029,0.533968
30,"{'penalty': 'l1', 'C': 30, 'class_weight': None, 'threshold': 0.7}",0.721569,0.258431,0.530653
43,"{'penalty': 'l1', 'C': 30, 'class_weight': None, 'threshold': 0.7}",0.721569,0.258431,0.530653


In [14]:
fairness_random_search.cv_results_.tail()

,params,predictive-score,fairness-score,combined-score
17,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
33,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
24,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
4,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
29,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.6}",0.466947,0.065593,0.385837


In [15]:
fairness_random_search.best_params_

{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced', 'threshold': 0.6}

In [16]:
fairness_random_search.best_score_

0.7319327731092438

In [17]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='combined', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=50, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [18]:
fairness_random_search.cv_results_.head()

,params,predictive-score,fairness-score,combined-score
34,"{'penalty': 'l2', 'C': 30, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
19,"{'penalty': 'l2', 'C': 75, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
38,"{'penalty': 'l2', 'C': 30, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
15,"{'penalty': 'l2', 'C': 0.01, 'class_weight': 'balanced', 'threshold': 0.3}",0.556303,0.044634,0.590921
17,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368


In [19]:
fairness_random_search.cv_results_.tail()

,params,predictive-score,fairness-score,combined-score
26,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.8}",0.608683,0.175782,0.461499
2,"{'penalty': 'l2', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.4}",0.707283,0.283003,0.460931
31,"{'penalty': 'l1', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.4}",0.709804,0.287279,0.458245
18,"{'penalty': 'l1', 'C': 100, 'class_weight': 'balanced', 'threshold': 0.4}",0.709804,0.287279,0.458245
29,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.6}",0.466947,0.065593,0.385837


In [20]:
fairness_random_search.best_params_

{'penalty': 'l2', 'C': 75, 'class_weight': None, 'threshold': 0.2}

In [21]:
fairness_random_search.best_score_

0.6019046698635167

- **Testing if the rest of the metrics work (from a programming point of view):**

In [22]:
fairness_metrics = ['statistical_parity_difference', 'abs_statistical_parity_difference',
                    'disparate_impact_ratio', 'abs_equal_opportunity_difference',
                    'average_odds_error', 'false_positive_rate_difference', 
                    'false_negative_rate_difference', 'true_positive_rate_difference',
                    'true_negative_rate_difference', 'false_positive_rate_ratio',
                    'false_negative_rate_ratio', 'true_positive_rate_ratio', 
                    'true_negative_rate_ratio', 'positive_predicted_value_difference',
                    'positive_predicted_value_ratio', 'positive_predicted_value_abs_difference']

In [23]:
best_score, best_params = {}, {}

for fairness_metric in fairness_metrics:
    print(fairness_metric)

    fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                        fairness_scoring=fairness_metric, 
                                                        predictive_scoring='balanced_accuracy',
                                                        objective='combined', 
                                                        fairness_scoring_direction='minimize',
                                                        predictive_scoring_direction='maximize',
                                                        fairness_weight=0.5, predictive_weight=0.5,
                                                        cv=inner, n_iter=20, random_state=123,
                                                        sens_variable_name=sens_variable_name, 
                                                        priv_group=sens_privileged_groups[0][sens_variable_name],
                                                        pos_label=response_favorable_label)

    fairness_random_search.fit(X=X_train, y=Y_train)

    display(fairness_random_search.cv_results_.head(3))
    best_score[fairness_metric] = fairness_random_search.best_score_
    best_params[fairness_metric] = fairness_random_search.best_params_

statistical_parity_difference


,params,predictive-score,fairness-score,combined-score
7,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced', 'threshold': 0.6}",0.731933,-0.265248,0.961656
13,"{'penalty': 'l2', 'C': 1, 'class_weight': None, 'threshold': 0.7}",0.717087,-0.278647,0.952972
18,"{'penalty': 'l1', 'C': 100, 'class_weight': 'balanced', 'threshold': 0.4}",0.709804,-0.287279,0.952295


abs_statistical_parity_difference


,params,predictive-score,fairness-score,combined-score
19,"{'penalty': 'l2', 'C': 75, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.559046
15,"{'penalty': 'l2', 'C': 0.01, 'class_weight': 'balanced', 'threshold': 0.3}",0.556303,0.044634,0.543693
7,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced', 'threshold': 0.6}",0.731933,0.265248,0.538344


disparate_impact_ratio


...

,params,predictive-score,fairness-score,combined-score
1,"{'penalty': 'l2', 'C': 0.1, 'class_weight': 'balanced', 'threshold': 0.8}",0.632773,-0.364925,0.772707
0,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced', 'threshold': 0.8}",0.672269,-0.258327,0.715748
9,"{'penalty': 'l1', 'C': 30, 'class_weight': None, 'threshold': 0.9}",0.671148,-0.260048,0.715627


positive_predicted_value_ratio
0.8736263736263735
0.0
1.090909090909091
1.0727272727272728
0.5326086956521738
1.0
0.0
0.0
1.0256410256410258
1.0
0.7105263157894737
0.890652557319224
0.9583333333333335
1.2473118279569892
0.7293156281920327
0.7124505928853756
0.8714932126696833
0.8970588235294117
1.2959183673469388
0.6618589743589743
0.7473118279569892
0.783839212469237
0.763888888888889
0.896969696969697
0.7544604927782497
0.8915094339622641
1.0888888888888888
1.0769230769230769
1.0819672131147542
0.5188679245283019
0.7156565656565657
0.837037037037037
0.7659313725490197
1.2815533980582525
0.6689976689976689
0.8903743315508021
0.8671875
1.12987012987013
1.1710526315789473
0.8209876543209877
0.8746355685131195
0.0
1.0714285714285714
1.0555555555555556
0.6976744186046511
0.8921568627450981
0.0
1.0952380952380953
1.0566037735849056
0.5222222222222221
0.8736263736263735
0.0
1.090909090909091
1.0727272727272728
0.5326086956521738
0.6916666666666667
0.819047619047619
1.0241477272727273
1.2261

,params,predictive-score,fairness-score,combined-score
0,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced', 'threshold': 0.8}",0.672269,0.713974,0.505951
9,"{'penalty': 'l1', 'C': 30, 'class_weight': None, 'threshold': 0.9}",0.671148,0.713244,0.503910
10,"{'penalty': 'l1', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.8}",0.670308,0.713974,0.501724


positive_predicted_value_abs_difference


,params,predictive-score,fairness-score,combined-score
7,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced', 'threshold': 0.6}",0.731933,0.129241,0.832951
13,"{'penalty': 'l2', 'C': 1, 'class_weight': None, 'threshold': 0.7}",0.717087,0.159653,0.761637
11,"{'penalty': 'l2', 'C': 100, 'class_weight': None, 'threshold': 0.7}",0.714566,0.170469,0.742222


In [24]:
best_score

{'statistical_parity_difference': 0.9616557334863584,
 'abs_statistical_parity_difference': 0.5590455937637937,
 'disparate_impact_ratio': 0.7437987716550986,
 'abs_equal_opportunity_difference': 0.5876190305476434,
 'average_odds_error': 0.6353113308412655,
 'false_positive_rate_difference': 0.9679951690821256,
 'false_negative_rate_difference': 0.6270253208482159,
 'true_positive_rate_difference': 0.9385217516970483,
 'true_negative_rate_difference': 0.8047434505530842,
 'false_positive_rate_ratio': 0.786231884057971,
 'false_negative_rate_ratio': 0.8672901535536762,
 'true_positive_rate_ratio': 0.7085232707692141,
 'true_negative_rate_ratio': 0.609743153057694,
 'positive_predicted_value_difference': 0.7727074549867866,
 'positive_predicted_value_ratio': 0.505951362341991,
 'positive_predicted_value_abs_difference': 0.8329512588389749}

In [25]:
best_params

{'statistical_parity_difference': {'penalty': 'l1',
  'C': 1,
  'class_weight': 'balanced',
  'threshold': 0.6},
 'abs_statistical_parity_difference': {'penalty': 'l2',
  'C': 75,
  'class_weight': None,
  'threshold': 0.2},
 'disparate_impact_ratio': {'penalty': 'l2',
  'C': 0.1,
  'class_weight': None,
  'threshold': 0.8},
 'abs_equal_opportunity_difference': {'penalty': 'l2',
  'C': 75,
  'class_weight': None,
  'threshold': 0.2},
 'average_odds_error': {'penalty': 'l1',
  'C': 1,
  'class_weight': 'balanced',
  'threshold': 0.6},
 'false_positive_rate_difference': {'penalty': 'l2',
  'C': 1,
  'class_weight': None,
  'threshold': 0.7},
 'false_negative_rate_difference': {'penalty': 'l2',
  'C': 75,
  'class_weight': None,
  'threshold': 0.2},
 'true_positive_rate_difference': {'penalty': 'l2',
  'C': 0.1,
  'class_weight': None,
  'threshold': 0.8},
 'true_negative_rate_difference': {'penalty': 'l2',
  'C': 0.1,
  'class_weight': 'balanced',
  'threshold': 0.7},
 'false_positive_ra